### Function Calling

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data,build_index

In [1]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [1]:
from ingest import load_faq_data,build_index
documents = load_faq_data()
index= build_index(documents)

In [2]:
index.search('How do I run Olama')

[{'id': '122d2b0aed',
  'course': 'data-engineering-zoomcamp',
  'section': 'Workshop 1 - dlthub',
  'question': 'How do I install the necessary dependencies to run the code?',
  'answer': 'To run the provided code, ensure that the `dlt[duckdb]` package is installed. You can do this by executing the following installation command in a Jupyter notebook:\n\n```bash\n!pip install dlt[duckdb]\n```\n\nIf you’re installing it locally, make sure to also have `duckdb` installed before the `duckdb` package is loaded:\n\n```zsh\npip install "dlt[duckdb]"\n```'},
 {'id': 'fdcf991009',
  'course': 'machine-learning-zoomcamp',
  'section': 'Module 9. Serverless Deep Learning',
  'question': 'Docker: run error',
  'answer': '```\ndocker: Error response from daemon: mkdir /var/lib/docker/overlay2/37be849565da96ac3fce34ee9eb2215bd6cd7899a63ebc0ace481fd735c4cb0e-init: read-only file system.\n```\n\nTo resolve this error, restart the Docker services.'},
 {'id': 'de0726d5ea',
  'course': 'machine-learnin

In [3]:
import os
from openai import OpenAI

# Configuración para usar Gemini a través de la API de compatibilidad de OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [4]:
messages = [
    {"role": "user", "content": "How do I run Olama locally?"}
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    
)
response.choices[0].message.content

'Running Ollama locally is a straightforward process, and it allows you to download, run, and interact with large language models (LLMs) directly on your computer, often leveraging your GPU for acceleration.\n\nHere\'s a comprehensive guide to get you started:\n\n---\n\n### Why Run Ollama Locally?\n\n*   **Privacy:** Your data stays on your machine.\n*   **Offline Access:** Run models without an internet connection (after initial download).\n*   **Cost-Effective:** No API fees, just your local hardware.\n*   **Customization:** Easily experiment with different models, create your own, or fine-tune existing ones.\n*   **Speed:** Depending on your hardware, local inference can be very fast.\n\n### Prerequisites\n\n*   **Operating System:** macOS (Apple Silicon or Intel), Windows 10/11, or Linux.\n*   **RAM:** At least 8 GB, but 16 GB+ is highly recommended for larger models.\n*   **GPU (Recommended):** An NVIDIA GPU (with CUDA support) or an AMD GPU (Linux only for now) will significantly

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [7]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]  
)


In [8]:
call  = response.choices[0].message.tool_calls[0]

In [9]:
call

ChatCompletionMessageFunctionToolCall(id='function-call-106048192043665178', function=Function(arguments='{"query":"run Olama locally"}', name='search'), type='function')

In [10]:
import json
args = json.loads(call.function.arguments)


In [11]:
call.function.name

'search'

In [12]:
results = search(**args)

In [13]:
result_json= results

In [41]:
result_json

'[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL https://ollama.com/install.sh | sh\\n  ```\\n\\nOnce installed, open a terminal and type:\\n\\n```bash\\nollama run llama3\\n```\\n\\nThis command will:\\n\\n- Download the LLaMA 3 model (~4GB).\\n- Start the model locally.\\n- Open a chat-like interface where you can type questions.\\n\\nTo test the Ollama local server, run the following command:\\n\\n```bash\\ncurl http://localhost:11434\\n```\\n\\nYou should receive a response similar to:\\n\\n```json\\n{\\"models\\": [...]}  \\n```\\n\\nThen, install th

In [14]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.id,
    "output": result_json,
}

In [15]:
messages.append(call)

In [16]:
messages.append(function_call_output)

In [17]:
messages

[{'role': 'user', 'content': 'How do I run Olama locally?'},
 ChatCompletionMessageFunctionToolCall(id='function-call-106048192043665178', function=Function(arguments='{"query":"run Olama locally"}', name='search'), type='function'),
 {'type': 'function_call_output',
  'call_id': 'function-call-106048192043665178',
  'output': [{'id': 'aa310de435',
    'course': 'llm-zoomcamp',
    'section': 'Module 1: RAG',
    'question': 'Can I run the course locally instead of Codespaces?',
    'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
   {'id': 'e8df9f0d12',
    'course': 'llm-zoomcamp',
    'section': 'Module 6: Best Practices',
    'question': 'Docker: When trying to run a streamlit app using docker-

In [19]:
messages = [
    {"role": "user", "content": "How do I run Ollama locally?"},
    {"role": "assistant", "tool_calls": [call]},  # del response anterior
    {
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(results)
    }
]

In [20]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]  
)


In [ ]:
print(response.choices[0].message.content)

In [25]:
response.usage.completion_tokens

376

In [28]:
def calculate_cost(prompt_tokens, completion_tokens):
    input_cost  = (prompt_tokens     / 1_000_000) * 0.30
    output_cost = (completion_tokens / 1_000_000) * 2.50
    total_cost       = input_cost + output_cost


    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

# Ejemplo
calculate_cost(1280, 376)

{'input_cost': 0.000384, 'output_cost': 0.00094, 'total_cost': 0.001324}

In [29]:
result = calculate_cost(response.usage.prompt_tokens, response.usage.completion_tokens)
print("Total cost: $", round(result["total_cost"], 8))


Total cost: $ 0.001324


### The Agentic Loop


In [92]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [93]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [68]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]  
)

In [72]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ('content', None),
 ('refusal', None),
 ('role', 'assistant'),
 ('annotations', None),
 ('audio', None),
 ('function_call', None),
 ('tool_calls',
  [ChatCompletionMessageFunctionToolCall(id='function-call-2974167831716732665', function=Function(arguments='{"query":"join course"}', name='search'), type='function'),
   ChatCompletionMessageFunctionToolCall(id='function-call-2974167831716732662', function=Funct

In [88]:
response.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='function-call-2974167831716732665', function=Function(arguments='{"query":"join course"}', name='search'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='function-call-2974167831716732662', function=Function(arguments='{"query":"enroll course"}', name='search'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='function-call-2974167831716732659', function=Function(arguments='{"query":"late enrollment"}', name='search'), type='function')]

In [94]:
messages.extend(response.choices[0].message.tool_calls)

for item in response.choices[0].message.tool_calls:
    if item.type == 'function':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

AttributeError: 'ChatCompletionMessageFunctionToolCall' object has no attribute 'name'

In [91]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ('content', None),
 ('refusal', None),
 ('role', 'assistant'),
 ('annotations', None),
 ('audio', None),
 ('function_call', None),
 ('tool_calls',
  [ChatCompletionMessageFunctionToolCall(id='function-call-2974167831716732665', function=Function(arguments='{"query":"join course"}', name='search'), type='function'),
   ChatCompletionMessageFunctionToolCall(id='function-call-2974167831716732662', function=Funct

In [58]:
call  = response.choices[0].message

In [89]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = result

    return {
        "type": "function_call_output",
        "call_id": call.id,
        "output": result_json,
    }

In [59]:
messages.extend(call)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

AttributeError: 'ChatCompletion' object has no attribute 'output'

In [45]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]  
)






In [48]:
response

ChatCompletion(id='jkQ7aoPLN-3NjMcPhf6d6A8', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-9608086107307222797', function=Function(arguments='{"query":"join course"}', name='search'), type='function'), ChatCompletionMessageFunctionToolCall(id='function-call-9608086107307222722', function=Function(arguments='{"query":"enroll course"}', name='search'), type='function'), ChatCompletionMessageFunctionToolCall(id='function-call-9608086107307222647', function=Function(arguments='{"query":"late enrollment"}', name='search'), type='function')]))], created=1782269071, model='gemini-2.5-flash', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=42, prompt_tokens=166, total_tokens=271, completion_tokens_detail

In [37]:
print(response.choices[0].message.content)

None


In [ ]:
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)